# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/dahye411/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "combo-hard-mining"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# Combo-Hard Mining:
#   (1) Set A train 에서 적게 나온 속성 조합을 찾고,
#   (2) 기존 best 모델이 Set B 에서 헷갈리는 정도를 계산한 뒤,
#   (3) 두 점수를 합쳐 가치가 높은 1,000장을 선택

# 기준:
#   score = 0.60 * 부족한 조합 점수 + 0.40 * 모델이 어려워한 정도

# 1단계 — best base 모델로 Set B 의 모든 이미지를 score.

import math
from collections import Counter

BASE_CKPT = "./pretrained_checkpoints/level3_focal_weather_sampler_mix.pth"
COMBO_WEIGHT = 0.60
HARD_WEIGHT = 0.40

if not os.path.exists(BASE_CKPT):
    raise FileNotFoundError(f"BASE_CKPT not found: {BASE_CKPT}.")

model = vit_small_patch16_224().to(device)
ckpt = torch.load(BASE_CKPT, map_location="cpu")
state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()

# Set A train에서 적게 등장한 조합일수록 높은 gap score를 할당
train_base = BDDAttrDataset("../data/set_a", "train", transform=None)

def sample_key(s):
    return int(s.weather), int(s.scene), int(s.timeofday)

cnt_w = Counter(int(s.weather) for s in train_base.samples)
cnt_s = Counter(int(s.scene) for s in train_base.samples)
cnt_t = Counter(int(s.timeofday) for s in train_base.samples)
cnt_ws = Counter((int(s.weather), int(s.scene)) for s in train_base.samples)
cnt_wt = Counter((int(s.weather), int(s.timeofday)) for s in train_base.samples)
cnt_st = Counter((int(s.scene), int(s.timeofday)) for s in train_base.samples)
cnt_wst = Counter(sample_key(s) for s in train_base.samples)


# count=0 이면 1.0, count 가 많아질수록 작아짐
# sqrt 를 써서 극소수 조합을 강하게 보상하되, 너무 급격하게 튀지 않도록 함
def gap(counter, key):
    return 1.0 / math.sqrt(counter.get(key, 0) + 1.0)

# weather/scene/timeofday의 조합의 부족이 성능 저하의 핵심임
# 3-way 조합을 가장 크게 보고, 2-way 조합을 보조로 사용
def combo_gap_score(w, s, t):
    return (
        0.55 * gap(cnt_wst, (w, s, t)) +
        0.20 * gap(cnt_wt, (w, t)) +
        0.15 * gap(cnt_ws, (w, s)) +
        0.10 * gap(cnt_st, (s, t))
    )

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

true_b = {
    "weather": np.array([s.weather for s in set_b.samples], dtype=np.int64),
    "scene": np.array([s.scene for s in set_b.samples], dtype=np.int64),
    "timeofday": np.array([s.timeofday for s in set_b.samples], dtype=np.int64),
}

# 정답 확률이 낮고, confidence 가 낮고, 예측이 틀릴수록 hard score 점수를 높게 할당
# Set B 라벨은 공개되어 있으므로 true label 을 직접 사용
hard_parts = []
model_error_count = np.zeros(len(set_b), dtype=np.int64)
for a in ATTRIBUTES:
    y = true_b[a]
    if np.any(y < 0):
        raise ValueError("Set B labels are required for Combo-Hard Mining, but some labels are missing.")
    p = probs_b[a]
    p_true = p[np.arange(len(y)), y]
    ce = -np.log(np.clip(p_true, 1e-8, 1.0)) / np.log(NUM_CLASSES[a])
    ce = np.clip(ce, 0.0, 2.0) / 2.0

    wrong = (preds_b[a] != y).astype(np.float32)
    uncertainty = 1.0 - p.max(axis=1)

    model_error_count += wrong.astype(np.int64)
    hard_parts.append(0.55 * ce + 0.30 * wrong + 0.15 * uncertainty)

hard_score = np.mean(np.stack(hard_parts, axis=1), axis=1)

combo_score = np.array([
    combo_gap_score(int(s.weather), int(s.scene), int(s.timeofday))
    for s in set_b.samples
], dtype=np.float32)

# 두 점수를 0~1 범위로 맞춘 뒤 결합
def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

combo_score = minmax(combo_score)
hard_score = minmax(hard_score)
final_score = COMBO_WEIGHT * combo_score + HARD_WEIGHT * hard_score

print(f"Set B size: {len(set_b)}")
print(f"combo score: min={combo_score.min():.3f}, mean={combo_score.mean():.3f}, max={combo_score.max():.3f}")
print(f"hard  score: min={hard_score.min():.3f}, mean={hard_score.mean():.3f}, max={hard_score.max():.3f}")
print(f"final score: min={final_score.min():.3f}, mean={final_score.mean():.3f}, max={final_score.max():.3f}")

del model
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# 2단계 — 본인의 선별 알고리즘을 설계하세요.

import json

K = 1000
MAX_PER_COMBO = 80           # 같은 weather-scene-timeofday 조합이 80장을 넘지 않도록 제한
MAX_PER_WEATHER = 360        # clear/overcast 같은 다수 weather 로 쏠리는 것을 방지
MAX_PER_SCENE = 520
MAX_PER_TIME = 620

order = np.argsort(-final_score)
selected = []
selected_set = set()
pick_cnt_w = Counter()
pick_cnt_s = Counter()
pick_cnt_t = Counter()
pick_cnt_wst = Counter()
fallback_fill_count = 0

for i in order:
    s = set_b.samples[int(i)]
    w, scene, t = int(s.weather), int(s.scene), int(s.timeofday)
    combo = (w, scene, t)

    if pick_cnt_wst[combo] >= MAX_PER_COMBO:
        continue
    if pick_cnt_w[w] >= MAX_PER_WEATHER:
        continue
    if pick_cnt_s[scene] >= MAX_PER_SCENE:
        continue
    if pick_cnt_t[t] >= MAX_PER_TIME:
        continue

    selected.append(int(i))
    selected_set.add(int(i))
    pick_cnt_w[w] += 1
    pick_cnt_s[scene] += 1
    pick_cnt_t[t] += 1
    pick_cnt_wst[combo] += 1

    if len(selected) == K:
        break

# cap 때문에 1,000장이 안 채워지면 남은 것은 score 순으로 채움
if len(selected) < K:
    for i in order:
        i = int(i)
        if i in selected_set:
            continue
        s = set_b.samples[i]
        w, scene, t = int(s.weather), int(s.scene), int(s.timeofday)
        combo = (w, scene, t)
        selected.append(i)
        selected_set.add(i)
        pick_cnt_w[w] += 1
        pick_cnt_s[scene] += 1
        pick_cnt_t[t] += 1
        pick_cnt_wst[combo] += 1
        fallback_fill_count += 1
        if len(selected) == K:
            break

assert len(selected) == K, f"selected {len(selected)} samples, expected {K}"

picks = []
for rank, i in enumerate(selected, start=1):
    s = set_b.samples[i]
    w, scene, t = int(s.weather), int(s.scene), int(s.timeofday)
    combo = (w, scene, t)
    picks.append({
        "rank": rank,
        "image_id": s.image_id,
        "weather": w,
        "scene": scene,
        "timeofday": t,
        "weather_name": CLASS_NAMES["weather"][w],
        "scene_name": CLASS_NAMES["scene"][scene],
        "timeofday_name": CLASS_NAMES["timeofday"][t],
        "score": float(final_score[i]),
        "combo_gap_score": float(combo_score[i]),
        "hard_score": float(hard_score[i]),
        "set_a_combo_count": int(cnt_wst.get(combo, 0)),
        "model_error_count": int(model_error_count[i]),
        "reason": (
            "Combo-Hard: Set A에서 부족한 weather-scene-timeofday 조합이면서 baseline 모델의 confidence/정답확률/오분류 기준으로 어려운 샘플"
        ),
    })

with open("../level5_picks.json", "w", encoding="utf-8") as f:
    json.dump({
        "strategy": "Combo-Hard Mining: score = 0.60 * Set A rare-combination gap + 0.40 * baseline hard-example score",
        "num_picks": len(picks),
        "fallback_fill_count": fallback_fill_count,
        "selection_constraints": {
            "max_per_weather_scene_timeofday_combo": MAX_PER_COMBO,
            "max_per_weather": MAX_PER_WEATHER,
            "max_per_scene": MAX_PER_SCENE,
            "max_per_timeofday": MAX_PER_TIME,
        },
        "score_weights": {"combo_gap": COMBO_WEIGHT, "hard_example": HARD_WEIGHT},
        "picks": picks,
    }, f, indent=2, ensure_ascii=False)

picks_for_training = picks
assert len(picks_for_training) == 1000

print(f"saved {len(picks)} picks -> ../level5_picks.json")
print(f"fallback fill count: {fallback_fill_count}")
print(f"training subset: {len(picks_for_training)} curated picks")
print("weather distribution:", {CLASS_NAMES["weather"][k]: v for k, v in Counter(p["weather"] for p in picks).items()})
print("scene distribution:", {CLASS_NAMES["scene"][k]: v for k, v in Counter(p["scene"] for p in picks).items()})
print("timeofday distribution:", {CLASS_NAMES["timeofday"][k]: v for k, v in Counter(p["timeofday"] for p in picks).items()})
print("top 10 combo counts:", pick_cnt_wst.most_common(10))


In [ ]:
# Curation Report용 picks 분포 시각화.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPORT_DIR = "../level5_report_artifacts"
os.makedirs(REPORT_DIR, exist_ok=True)

pick_rows = []
selected_indices = [selected_rank_idx for selected_rank_idx in selected]
for p, i in zip(picks, selected_indices):
    row = dict(p)
    for a in ATTRIBUTES:
        row[f"pred_{a}"] = int(preds_b[a][i])
        row[f"pred_{a}_name"] = CLASS_NAMES[a][int(preds_b[a][i])]
    pick_rows.append(row)

picks_df = pd.DataFrame(pick_rows)
picks_df.to_csv(f"{REPORT_DIR}/combo_hard_picks_distribution_table.csv", index=False)

def plot_combo_heatmaps(df, prefix="label", out_name="picks_label_combo_heatmap.png"):
    w_col = "weather_name" if prefix == "label" else "pred_weather_name"
    s_col = "scene_name" if prefix == "label" else "pred_scene_name"
    t_col = "timeofday_name" if prefix == "label" else "pred_timeofday_name"

    scenes = CLASS_NAMES["scene"]
    fig, axes = plt.subplots(1, len(scenes), figsize=(5 * len(scenes), 4), sharey=True)
    if len(scenes) == 1:
        axes = [axes]

    for ax, scene_name in zip(axes, scenes):
        sub = df[df[s_col] == scene_name]
        table = pd.crosstab(sub[w_col], sub[t_col]).reindex(
            index=CLASS_NAMES["weather"], columns=CLASS_NAMES["timeofday"], fill_value=0
        )
        sns.heatmap(table, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
        ax.set_title(scene_name)
        ax.set_xlabel("timeofday")
        ax.set_ylabel("weather")

    fig.suptitle(f"Combo-Hard picks distribution ({prefix})", y=1.03)
    fig.tight_layout()
    out_path = f"{REPORT_DIR}/{out_name}"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    return out_path

def plot_stacked_bars(df, prefix="label", out_name="picks_label_stacked_bar.png"):
    w_col = "weather_name" if prefix == "label" else "pred_weather_name"
    s_col = "scene_name" if prefix == "label" else "pred_scene_name"
    t_col = "timeofday_name" if prefix == "label" else "pred_timeofday_name"

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ws = pd.crosstab(df[w_col], df[s_col]).reindex(
        index=CLASS_NAMES["weather"], columns=CLASS_NAMES["scene"], fill_value=0
    )
    ws.plot(kind="bar", stacked=True, ax=axes[0])
    axes[0].set_title(f"weather x scene ({prefix})")
    axes[0].set_xlabel("weather")
    axes[0].set_ylabel("count")
    axes[0].tick_params(axis="x", rotation=35)

    wt = pd.crosstab(df[w_col], df[t_col]).reindex(
        index=CLASS_NAMES["weather"], columns=CLASS_NAMES["timeofday"], fill_value=0
    )
    wt.plot(kind="bar", stacked=True, ax=axes[1])
    axes[1].set_title(f"weather x timeofday ({prefix})")
    axes[1].set_xlabel("weather")
    axes[1].set_ylabel("count")
    axes[1].tick_params(axis="x", rotation=35)

    fig.tight_layout()
    out_path = f"{REPORT_DIR}/{out_name}"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    return out_path

label_heatmap_path = plot_combo_heatmaps(picks_df, prefix="label", out_name="picks_label_weather_scene_timeofday_heatmap.png")
pred_heatmap_path = plot_combo_heatmaps(picks_df, prefix="pred", out_name="picks_pred_weather_scene_timeofday_heatmap.png")
label_bar_path = plot_stacked_bars(picks_df, prefix="label", out_name="picks_label_stacked_bar.png")
pred_bar_path = plot_stacked_bars(picks_df, prefix="pred", out_name="picks_pred_stacked_bar.png")

print("saved report figures:")
for p in [label_heatmap_path, pred_heatmap_path, label_bar_path, pred_bar_path]:
    print(" -", p)


In [ ]:
# 3단계 — Set A + 본인이 고른 picks 로 재학습. 학습 메트릭은 wandb 로 자동 로깅.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPORT_DIR = "../level5_report_artifacts"
os.makedirs(REPORT_DIR, exist_ok=True)

LEVEL5_EPOCHS = 30
VARIANT_TO_RUN = "combo1000"   # "random1000", "combo250", "combo500", "combo1000" 중 하나
RUN_TRAINING = True            # 이미 결과 CSV/checkpoint가 있으면 False로 두고 요약 그림만 재생성 가능

rng = np.random.default_rng(SEED)
random_indices = rng.choice(len(set_b.samples), size=1000, replace=False).tolist()


def make_records_from_indices(indices, source):
    records = []
    for rank, i in enumerate(indices, start=1):
        s = set_b.samples[int(i)]
        records.append({
            "rank": rank,
            "image_id": s.image_id,
            "weather": int(s.weather),
            "scene": int(s.scene),
            "timeofday": int(s.timeofday),
            "source": source,
        })
    return records

variant_picks_map = {
    "random1000": make_records_from_indices(random_indices, "random1000"),
    "combo250": picks[:250],
    "combo500": picks[:500],
    "combo1000": picks[:1000],
}

if VARIANT_TO_RUN not in variant_picks_map:
    raise ValueError(f"Unknown VARIANT_TO_RUN={VARIANT_TO_RUN}. Choose one of {list(variant_picks_map)}")


def train_level5_variant(variant_name, variant_picks, epochs=LEVEL5_EPOCHS):
    extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in variant_picks]
    train_aug = BDDAttrDataset("../data/set_a", "train", transform=train_transform(), extra_picks=extra)
    val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())

    sampler = class_balanced_sampler(train_aug, attribute="weather")
    loader_tr = DataLoader(
        train_aug, batch_size=64, sampler=sampler, num_workers=2,
        worker_init_fn=seed_worker, pin_memory=True,
    )
    loader_val = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    set_seed(SEED, deterministic=True)
    model_variant = vit_small_patch16_224().to(device)
    model_variant.load_state_dict(state, strict=True)

    optim = torch.optim.AdamW(model_variant.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: FocalLoss(gamma=2.0).to(device) for a in ATTRIBUTES}

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level5-{STRATEGY_NAME}-{variant_name}",
        config={
            "backbone": "vit_s16",
            "init": "imagenet_pretrained_level3_ckpt",
            "init_ckpt": BASE_CKPT,
            "strategy": STRATEGY_NAME,
            "variant": variant_name,
            "num_picks": len(variant_picks),
            "loss": "focal",
            "sampler": "class_balanced(weather)",
            "epochs": epochs,
            "batch": 64,
            "lr": 5e-4,
            "weight_decay": 5e-2,
            "seed": SEED,
        },
        tags=WANDB_TAGS + [variant_name],
    )

    trainer = MultiTaskTrainer(
        model_variant, optim, sched, losses, device,
        TrainConfig(epochs=epochs, amp=(device.type == "cuda")), logger=logger,
    )
    history = trainer.fit(loader_tr, loader_val)

    val_pred, _, val_tgt, _ = collect_predictions(model_variant, loader_val, device)
    final_metrics = trainer.evaluate(loader_val)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])

    best_epoch = int(np.argmax(history["val_avg_mf1"]) + 1)
    best_avg_mf1 = float(np.max(history["val_avg_mf1"]))
    final_avg_mf1 = float(final_metrics["avg_macro_f1"])

    result = {
        "variant": variant_name,
        "num_picks": len(variant_picks),
        "best_epoch": best_epoch,
        "best_avg_mf1": best_avg_mf1,
        "final_avg_mf1": final_avg_mf1,
    }
    for a, v in final_metrics["per_macro_f1"].items():
        result[f"final_mf1_{a}"] = float(v)

    logger.log({
        "summary/best_avg_mf1": best_avg_mf1,
        "summary/final_avg_mf1": final_avg_mf1,
    })
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    ckpt_path = f"../checkpoints/level5_{STRATEGY_NAME}_{variant_name}.pth"
    torch.save({
        "state_dict": model_variant.state_dict(),
        "history": history,
        "result": result,
        "num_picks": len(variant_picks),
    }, ckpt_path)
    result["checkpoint"] = ckpt_path
    return model_variant, history, result

combined_results_path = f"{REPORT_DIR}/level5_random_di_ablation_results.csv"
variant_result_paths = {
    name: f"{REPORT_DIR}/level5_result_{name}.csv"
    for name in variant_picks_map
}
model2 = None

if RUN_TRAINING:
    print(f"\n=== Running {VARIANT_TO_RUN}: {len(variant_picks_map[VARIANT_TO_RUN])} picks ===")
    model_variant, history, result = train_level5_variant(VARIANT_TO_RUN, variant_picks_map[VARIANT_TO_RUN])

    # 각 run 결과는 개별 CSV로 저장
    current_result_path = variant_result_paths[VARIANT_TO_RUN]
    pd.DataFrame([result]).to_csv(current_result_path, index=False)
    print(f"saved individual result: {current_result_path}")

    if VARIANT_TO_RUN == "combo1000":
        model2 = model_variant
    else:
        del model_variant
        if device.type == "cuda":
            torch.cuda.empty_cache()

# DI / ablation 계산은 개별 CSV들을 읽어서 수행
result_frames = []
for variant_name, result_path in variant_result_paths.items():
    if os.path.exists(result_path):
        result_frames.append(pd.read_csv(result_path))

if not result_frames:
    raise FileNotFoundError(
        "No individual result CSVs found. Run one variant first, e.g. VARIANT_TO_RUN='random1000'."
    )

results_df = pd.concat(result_frames, ignore_index=True)
results_df = results_df.drop_duplicates(subset=["variant"], keep="last")
results_df = results_df.sort_values("num_picks")

# 편의를 위해 개별 CSV들을 합친 요약 CSV도 매번 재생성
results_df.to_csv(combined_results_path, index=False)
print(f"saved combined summary: {combined_results_path}")

combo1000_ckpt_path = f"../checkpoints/level5_{STRATEGY_NAME}_combo1000.pth"
if model2 is None and os.path.exists(combo1000_ckpt_path):
    model2 = vit_small_patch16_224().to(device)
    combo_ckpt = torch.load(combo1000_ckpt_path, map_location="cpu")
    model2.load_state_dict(combo_ckpt["state_dict"], strict=True)
    model2.eval()
    print(f"loaded combo1000 model for submission: {combo1000_ckpt_path}")

display(results_df)

combo_ablation = results_df[results_df["variant"].isin(["combo250", "combo500", "combo1000"])].sort_values("num_picks")
if len(combo_ablation):
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.plot(combo_ablation["num_picks"], combo_ablation["final_avg_mf1"], marker="o")
    ax.set_xlabel("number of curated picks")
    ax.set_ylabel("final val Avg-MF1")
    ax.set_title("Level 5 ablation: marginal utility of curated data")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    ablation_plot_path = f"{REPORT_DIR}/level5_ablation_curve.png"
    fig.savefig(ablation_plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", ablation_plot_path)
else:
    print("ablation curve needs at least one of combo250/combo500/combo1000 results")

compare_df = results_df[results_df["variant"].isin(["random1000", "combo1000"])]
if len(compare_df) == 2:
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(data=compare_df, x="variant", y="final_avg_mf1", ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel("final val Avg-MF1")
    ax.set_title("Random-1000 baseline vs Combo-Hard-1000")
    fig.tight_layout()
    di_plot_path = f"{REPORT_DIR}/level5_random_vs_combo1000.png"
    fig.savefig(di_plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", di_plot_path)

random_rows = results_df[results_df["variant"] == "random1000"]
combo_rows = results_df[results_df["variant"] == "combo1000"]
if len(random_rows) and len(combo_rows):
    random_mf1 = float(random_rows.iloc[0]["final_avg_mf1"])
    combo_mf1 = float(combo_rows.iloc[0]["final_avg_mf1"])
    di_score = (combo_mf1 - random_mf1) / random_mf1
    print(f"DI score = ({combo_mf1:.4f} - {random_mf1:.4f}) / {random_mf1:.4f} = {di_score:.4f}")
else:
    di_score = None
    print("DI score 계산에는 random1000과 combo1000 결과가 모두 필요")


In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성.
if model2 is None:
    combo1000_ckpt_path = f"../checkpoints/level5_{STRATEGY_NAME}_combo1000.pth"
    if not os.path.exists(combo1000_ckpt_path):
        raise RuntimeError("combo1000 모델이 없습니다. VARIANT_TO_RUN='combo1000'으로 3단계를 먼저 실행하세요.")
    model2 = vit_small_patch16_224().to(device)
    combo_ckpt = torch.load(combo1000_ckpt_path, map_location="cpu")
    model2.load_state_dict(combo_ckpt["state_dict"], strict=True)
    model2.eval()

test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")


## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.